# PIT WALL — stage-1 corpus build (GPU)

> **This notebook is a build tool. It is not the application.**
>
> PIT WALL is a **Next.js frontend** (`frontend/`) against a **FastAPI backend**
> (`backend/main.py`), plus a Gradio model backend on ZeroGPU (`space_live/`).
> Nothing in this notebook is required to run any of them.
>
> All it does is run stage-1 inference — ASR, prosody, text sentiment — over
> 2,042 radio clips on a GPU and write `backend/raw/*.raw.jsonl`. Stage 2
> (calibration, fusion, lap join, strategy) runs on CPU in seconds and stays in
> the repository. This notebook ships in neither the Docker image nor either
> Space.
>
> It **imports** `backend/pipeline/*` rather than reimplementing it, so there is
> exactly one copy of every model call. If this notebook and the app ever
> disagree, that is a bug in the clone step, not a fork to reconcile.

**Why GPU.** Stage 1 is ~25s per clip on CPU, so the corpus is an overnight run.
On a T4 it is well under an hour, which is what makes the v2 rebuild practical.

**What comes back.** A ~20 MB tarball of JSONL. The clips (321 MB) are fetched
here from the Hugging Face dataset and are never uploaded from your machine.

## 1 · Check the GPU

Stop here if this says CPU — the accelerator is the whole point.

In [1]:
# On a CPU-only runtime `nvidia-smi` is not installed at all, so probing it# directly raises FileNotFoundError and buries the real problem in a# traceback. Check for the binary first, and say what to do about it.import shutilimport subprocessif shutil.which("nvidia-smi") is None:    raise SystemExit(        "No GPU is attached to this runtime.\n"        "\n"        "  VS Code: command palette -> Colab: Change Runtime Type -> T4 GPU,\n"        "           or disconnect and reconnect picking a GPU runtime.\n"        "  Browser: Runtime > Change runtime type > T4 GPU.\n"        "\n"        "Stage 1 is about 25s per clip on CPU, so the corpus would take\n"        "roughly 14 hours here against about 40 minutes on a T4.\n"        "Not worth starting without the accelerator."    )print(subprocess.run(    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",     "--format=csv,noheader"],    capture_output=True, text=True).stdout.strip())

/bin/bash: line 1: nvidia-smi: command not found


FileNotFoundError: [Errno 2] No such file or directory: 'nvidia-smi'

## 2 · Install

Pinned to match `requirements.txt`. A `ctranslate2` build mismatching the runtime
CUDA is the most common faster-whisper breakage on Colab, so it is pinned and
smoke-tested in 2b before anything long runs.

In [ ]:
%pip install -q "transformers==5.15.0" "librosa==0.11.0" "soundfile==0.14.0" "faster-whisper==1.2.1" "ctranslate2==4.8.1" "duckdb==1.5.5" "fastf1==3.8.3" "scipy==1.16.3" "pandas==2.1.3"

import platform, torch, transformers, faster_whisper, ctranslate2
PROVENANCE = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "transformers": transformers.__version__,
    "faster_whisper": faster_whisper.__version__,
    "ctranslate2": ctranslate2.__version__,
}
assert torch.cuda.is_available(), "torch cannot see the GPU"
print(PROVENANCE)

### 2b · One-clip smoke test of the CUDA stack

Ten seconds, and it fails here rather than forty minutes in.

In [ ]:
import numpy as np
from faster_whisper import WhisperModel

_m = WhisperModel("tiny", device="cuda", compute_type="float16")
_segs, _info = _m.transcribe(np.zeros(16000, dtype=np.float32), vad_filter=True)
list(_segs)
del _m
print("ctranslate2 + CUDA OK")

## 3 · Get the source onto the runtime

Cells run on the **Colab runtime**, not on your machine — including when you
drive this from the VS Code extension. So the runtime needs its own copy of
`backend/pipeline/` and `backend/data/`.

Two paths, and the cell below takes whichever is available:

**Clone (preferred).** Pin a commit and the build manifest records the SHA, so
the artifact traces back to exactly the code that produced it.

**Upload (fallback).** Until the repository has a remote, run
`python backend/tools/pack_source.py` locally and upload the resulting
`pitwall_src.zip` — 0.8 MB, and it carries a content hash so provenance is
still checkable without a commit to name.

In [ ]:
REPO = "https://github.com/rogerdemello/pitwall.git"   # empty string to upload a bundle instead
COMMIT = "5bf701ebc0a7"   # pin to the exact source this build ran on

import os, subprocess, sys
if not os.path.exists("/content/pitwall"):
    subprocess.run(["git", "clone", REPO, "/content/pitwall"], check=True)
subprocess.run(["git", "-C", "/content/pitwall", "checkout", COMMIT], check=True)

SHA = subprocess.run(["git", "-C", "/content/pitwall", "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip()

sys.path.insert(0, "/content/pitwall/backend")
print("building from", SHA)

## 4 · Choose the models

`PITWALL_ASR_MODEL` selects the checkpoint *and* the backend: anything with
`large` in the name routes to faster-whisper automatically.

large-v3 rather than distil-large-v3 or turbo, because
`research/exp_prompting.py` measured distillation mangling exactly the rare
tokens this domain lives on — "DRS" became "the areas", "Hamilton's pitted"
became "I will turn to the pitted". That evidence is already in hand.

In [ ]:
import os
os.environ["PITWALL_ASR_MODEL"] = "openai/whisper-large-v3"
os.environ["HF_HOME"] = "/content/hf_cache"

from pipeline import asr, device, prosody, sentiment
print("asr     ", asr.MODEL_ID, "via", asr.BACKEND)
print("prosody ", prosody.MODEL_ID)
print("device  ", device.describe())
PROVENANCE.update({"asr_model": asr.MODEL_ID, "asr_backend": asr.BACKEND,
                   "prosody_model": prosody.MODEL_ID, "decode": asr.DECODE})

## 5 · Checkpoints on Drive

Clips are re-downloadable in three minutes; inference is not. So the journal
lives on Drive and the audio does not.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CKPT = "/content/drive/MyDrive/pitwall_v2"
os.makedirs(CKPT, exist_ok=True)
os.makedirs("/content/pitwall/backend/raw", exist_ok=True)

import glob, shutil
for f in glob.glob(CKPT + "/*.raw.jsonl"):
    shutil.copy(f, "/content/pitwall/backend/raw/")
    print("resuming", os.path.basename(f), sum(1 for _ in open(f)), "clips done")

## 6 · Fetch the clips

Straight from the Hub via DuckDB. Nothing is uploaded from your machine.

In [ ]:
%cd /content/pitwall
import json
from data import fetch_many

# Derived from what is actually built, not from the two hardcoded lists.
# SLATE + SEASON_2023 is 11 races; the corpus is 12. The showcase race
# (2021 Abu Dhabi) was fetched separately by fetch_race_clips.py and is in
# neither list, so rebuilding from those alone would leave it on v1 output -
# a mixed corpus, which silently breaks calibration comparability and every
# cross-race claim resting on it.
from pipeline.artifacts import race_ids
RACES = race_ids("backend/races") or sorted(
    set(fetch_many.SLATE) | set(fetch_many.SEASON_2023))
print(f"rebuilding {len(RACES)} races")
fetch_many.fetch(RACES)          # ~321 MB, about 3 minutes

total = 0
for r in RACES:
    mf = "backend/clips/" + r + "/manifest.json"
    n = len(json.load(open(mf))) if os.path.exists(mf) else 0
    total += n
    print("  %-34s %4d clips" % (r, n))
print("\n%d clips over %d races" % (total, len(RACES)))

## 7 · Smoke test: 20 clips, side by side with v1

**Read this before starting the long run.** It is the cell that saves an hour.
Look for transcripts that are longer on clips over 30s — v1 truncated there —
and near-silent clips coming back empty rather than as the word "you".

In [ ]:
from data import build_race
build_race.build(RACES[0], limit=20)

v2 = [json.loads(l) for l in open("backend/raw/" + RACES[0] + ".raw.jsonl") if l.strip()]
old = "backend/races/" + RACES[0] + ".json"
v1 = {m["id"]: m for m in json.load(open(old))["messages"]} if os.path.exists(old) else {}

for r in v2[:20]:
    a = v1.get(r["id"], {})
    flag = "   <-- v1 truncated this" if a.get("duration_s", 0) > 30 else ""
    print("%6.1fs  words %3d -> %3d   windows %2d  voiced %.2f%s" % (
        r.get("duration_s", 0), len(a.get("transcript", "").split()),
        len(r.get("transcript", "").split()), r.get("windows", 0),
        r.get("voiced_fraction", 0), flag))
    if r.get("transcript", "") != a.get("transcript", ""):
        print("         v1: %r" % a.get("transcript", "")[:90])
        print("         v2: %r" % r.get("transcript", "")[:90])

## 8 · The long run

Append-only JSONL, so a disconnect costs the current clip and nothing else.
Progress prints keep the session marked active — Colab idles out after ~90
minutes without output.

Races run longest-first so an OOM or a bad time estimate surfaces in minute one
rather than minute fifty.

In [ ]:
import time, traceback

order = sorted(RACES, key=lambda r: -len(json.load(open("backend/clips/" + r + "/manifest.json"))))
t0 = time.time()

for i, race in enumerate(order, 1):
    print("\n=== [%d/%d] %s ===" % (i, len(order), race), flush=True)
    try:
        build_race.build(race)
    except Exception:
        traceback.print_exc()
        continue
    src = "backend/raw/" + race + ".raw.jsonl"
    if os.path.exists(src):
        shutil.copy(src, CKPT)
        print("  checkpointed -> " + CKPT, flush=True)
    print("  elapsed %.1f min" % ((time.time() - t0) / 60), flush=True)

print("\nSTAGE 1 COMPLETE in %.1f min" % ((time.time() - t0) / 60))

## 9 · Validate before shipping

A bundle that fails these is not worth downloading.

In [ ]:
problems, summary = [], []
for path in sorted(glob.glob("backend/raw/*.raw.jsonl")):
    race = os.path.basename(path)[:-len(".raw.jsonl")]
    rows = [json.loads(l) for l in open(path) if l.strip()]
    manifest = json.load(open("backend/clips/" + race + "/manifest.json"))
    ok = [r for r in rows if "error" not in r]
    if len(ok) < len(manifest):
        problems.append("%s: only %d/%d clips" % (race, len(ok), len(manifest)))
    for r in ok:
        if r.get("arousal") is None:
            problems.append("%s: %s has no affect" % (race, r["id"]))
        if r.get("windows", 0) == 0 and r.get("voiced_fraction", 0) > 0:
            problems.append("%s: %s has speech but no windows" % (race, r["id"]))
    summary.append((race, len(ok),
                    sum(1 for r in ok if r.get("windows", 0) == 0),
                    sum(1 for r in ok if r.get("duration_s", 0) > 30)))

print("%-34s%7s%11s%10s" % ("race", "clips", "no-speech", "over 30s"))
for race, n, silent, long_ in summary:
    print("%-34s%7d%11d%10d" % (race, n, silent, long_))
print("\ntotal %d clips, %d flagged no-speech" % (
    sum(s[1] for s in summary), sum(s[2] for s in summary)))
assert not problems, problems[:10]
print("validation passed")

## 10 · Package and download

In [ ]:
import hashlib, tarfile

PROVENANCE["clips"] = sum(s[1] for s in summary)
PROVENANCE["races"] = len(summary)
with open("backend/raw/_manifest.json", "w") as fh:
    json.dump(PROVENANCE, fh, indent=1, default=str)

files = sorted(glob.glob("backend/raw/*.raw.jsonl")) + ["backend/raw/_manifest.json"]
with open("backend/raw/checksums.sha256", "w") as fh:
    for f in files:
        h = hashlib.sha256(open(f, "rb").read()).hexdigest()
        fh.write(h + "  " + os.path.basename(f) + "\n")
files.append("backend/raw/checksums.sha256")

with tarfile.open("/content/pitwall_raw_v2.tar.gz", "w:gz") as tar:
    for f in files:
        tar.add(f, arcname=os.path.basename(f))

print("pitwall_raw_v2.tar.gz  %.1f MB" % (os.path.getsize("/content/pitwall_raw_v2.tar.gz") / 1e6))
shutil.copy("/content/pitwall_raw_v2.tar.gz", CKPT)   # survives the session

from google.colab import files as colab_files
colab_files.download("/content/pitwall_raw_v2.tar.gz")

## 11 · Back on your machine

```bash
python backend/data/import_raw_bundle.py ~/Downloads/pitwall_raw_v2.tar.gz
python backend/data/finish_corpus.py          # pool + LORO calibration, re-apply
pytest backend/tests -q
```

`import_raw_bundle.py` verifies the checksums against the manifest, moves v1
aside to `backend/raw_v1/` so the paired v1-vs-v2 comparison stays possible, and
refuses to overwrite without `--force`.

Then, and only then, run the confirmatory test **once**:

```bash
python backend/data/corpus_analysis.py
```

Per `backend/races/_preregistration.json` that is a single confirmatory run, and
its result is reported whichever way it comes out.